### 1. Load the Glass Identification Dataset
We will load the dataset directly using `pandas` from the official UCI Repository.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data"
columns = ['Id', 'RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
df = pd.read_csv(url, names=columns)

# Drop Id column as it is not a feature
df = df.drop(columns=['Id'])

# Features and target split
X = df.drop(columns=['Type'])
y = df['Type']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale features (essential for KNN and helpful for stability overall)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset loaded. Training shape: {X_train.shape}, Test shape: {X_test.shape}")
display(df.head())

Dataset loaded. Training shape: (149, 9), Test shape: (65, 9)


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


### 2. Fit Models and Evaluate Accuracy
Now, we will train K-Nearest Neighbors, Naive Bayes, and Random Forest classifiers, and compare their overall accuracies.

In [3]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize models
knn = KNeighborsClassifier(n_neighbors=5)
nb = GaussianNB()
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit and predict
models = {
    "K-Nearest Neighbors": knn,
    "Naive Bayes": nb,
    "Random Forest": rf
}

results = {}
for name, model in models.items():
    # KNN performs better with scaled data, RF/NB can handle raw but we use scaled for consistent comparison
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    results[name] = acc
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, predictions, zero_division=0))
    print("\n")

=== K-Nearest Neighbors ===
Accuracy: 0.6769
              precision    recall  f1-score   support

           1       0.64      0.76      0.70        21
           2       0.68      0.74      0.71        23
           3       0.00      0.00      0.00         5
           5       0.50      0.25      0.33         4
           6       1.00      0.67      0.80         3
           7       0.73      0.89      0.80         9

    accuracy                           0.68        65
   macro avg       0.59      0.55      0.56        65
weighted avg       0.63      0.68      0.64        65



=== Naive Bayes ===
Accuracy: 0.5077
              precision    recall  f1-score   support

           1       0.47      0.86      0.61        21
           2       0.50      0.13      0.21        23
           3       0.00      0.00      0.00         5
           5       0.00      0.00      0.00         4
           6       0.43      1.00      0.60         3
           7       0.75      1.00      0.86     

### Conclusion and Recommendation
Based on typical runs of this dataset:
1. **Random Forest** generally performs the best (often achieving around 75-80% accuracy). It handles complex, non-linear relationships among physical/chemical properties of glass types effectively and is robust to noise and outliers.
2. **K-Nearest Neighbors** performs moderately well, but is highly sensitive to the choice of 'k' and distance metrics, and behaves less robustly on smaller datasets with highly unbalanced classes.
3. **Naive Bayes** usually performs the poorest here because it assumes features (such as refractive index and chemical concentrations like Al, Mg, Si) are independent, which is not true in chemical compositions.

Therefore, **Random Forest** is the most suitable model because it handles non-linear feature interactions without making unrealistic independence assumptions.